# 20 — Feature Engineering

**ENAI 603 Capstone — WMATA Metro Delay Prediction**

This notebook performs further feature engineering on the WMATA database with the aim of improving models performance.

**Sections:**
1. Load existing features.csv file
2. Add station has parking feature
3. Add transfer station feature
4. Add VRE, AMTRAK, MARK connection feature
5. Add Number of Rail systems connection (interaction term) feature
6. Save new features into existing features.csv

## 1. Load existing features.csv file

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path("..")

df_features = pd.read_csv(PROJECT_DIR / "data" / "features.csv")
df_features

---

# Feature Engineering

## 2. Add station has parking feature

In [ ]:
STATION_PARKING = ['N12', 'N11', 'N09', 'N08', 'N06', 'K05', 'K08', 'K07', 'K06', 'A15', 'A14', 'A13', 'A12', 'A11', 'J03', 'J02', 'C15',
                   'F11', 'F10', 'F09', 'F08', 'F06', 'G02', 'G03', 'G04', 'G05', 'D09', 'D10', 'D11', 'D12', 'D13',
                   'B04', 'B07', 'B06', 'B08', 'B09', 'B10', 'B11', 'E07', 'E08', 'E09', 'E10']

# Check if the upcoming/current station location has parking availability
def station_parking_available(location_code):
  if location_code in STATION_PARKING:
    return 1
  else:
    return 0

# Add loc_has_parking feature based on location_code
df_features['loc_has_parking'] = df_features['location_code'].apply(station_parking_available)
df_features

## 3. Add transfer station feature

In [ ]:
TRANSFER_STATIONS = ['K05', 'C05', 'C07', 'C13', 'A01', 'C01', 'B01', 'F01', 'D03', 'F03', 'E01', 'B06', 'E06' 'D08']

# Check if the upcoming/current station location is a transfer station
def is_transfer_station(location_code):
  if location_code in TRANSFER_STATIONS:
    return 1
  else:
    return 0

# Add loc_is_transfer feature based on location_code
df_features['loc_is_transfer'] = df_features['location_code'].apply(is_transfer_station)
df_features

## 4. Add VRE, AMTRAK, MARK connection feature

In [ ]:
RAIL_SYS_CONNECT = {
  'VRE': ['B03', 'C13', 'C09', 'J03', 'D03', 'F03'],
  'AMTRAK': ['D13', 'B03', 'C13', 'A14'],
  'MARK':  ['D13', 'E10', 'E09', 'B03', 'B08', 'A14']
}

# Check if the upcoming/current station location connects to any other rail systems
def connect_rail_systems(location_code):
  connects = []
  
  for rail, stations in RAIL_SYS_CONNECT.items():
    if location_code in stations:
      connects.append(1)
    else:
      connects.append(0)
  
  return connects

# Add VRE, AMTRAK, MARK connection feature based on location_code
df_features[['loc_conn_vre', 'loc_conn_amtrak', 'loc_conn_mark']] = df_features['location_code'].apply(connect_rail_systems).to_list()
df_features

## 5. Add Number of Rail systems connection feature

In [ ]:
# Add interaction term "num_rails_conn" based on sum of VRE, AMTRAK, MARK connections in a location
df_features['num_rails_conn'] = df_features['loc_conn_vre'] + df_features['loc_conn_amtrak'] + df_features['loc_conn_mark']
df_features[df_features['num_rails_conn'] > 1]

## 6. Save new features into existing features.csv

In [ ]:
df_features.to_csv(PROJECT_DIR / "data" / 'features.csv')